# 04 - Priority Prediction

Goal: train a separate priority classifier and add hybrid rule-based priority boosting.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from preprocessing import add_cleaned_ticket_column
from predict import detect_escalation_keywords, boost_priority
from train import build_vectorizer, get_candidate_models, RANDOM_STATE, TEST_SIZE
from utils import load_raw_data, prepare_ticket_dataframe

In [ ]:
df = add_cleaned_ticket_column(prepare_ticket_dataframe(load_raw_data()))
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_ticket'], df['priority'], test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=df['priority']
)
vectorizer = build_vectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [ ]:
models = get_candidate_models(include_naive_bayes=False)
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print('\n' + '=' * 80)
    print(name)
    print(classification_report(y_test, preds, zero_division=0))

In [ ]:
best_model_name = 'LinearSVC'
preds = models[best_model_name].predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, preds, xticks_rotation=45, cmap='Blues')
plt.title(f'Priority Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.show()

In [ ]:
ticket = 'Payment failed twice and refund is urgent because I cannot access my account'
ml_priority = 'Medium'
keywords = detect_escalation_keywords(ticket)
boosted = boost_priority(ml_priority, keywords)
print('Detected keywords:', keywords)
print('ML priority:', ml_priority)
print('Final boosted priority:', boosted)

## Why hybrid rules help

Businesses often use deterministic escalation rules alongside ML because some keywords indicate operational, financial, or security risk. Rules make those cases auditable and fast to adjust.